# Notebook 3 — Rerank RAG
**Livro:** Os Sertões — Euclides da Cunha  
**Estratégia:** Busca por embeddings (retrieval largo) → Cross-Encoder reranking → geração com os top-k rerankeados

## 1. Instalação de dependências

In [ ]:
!pip install -q anthropic pypdf chromadb sentence-transformers langchain langchain-community tiktoken

## 2. Importações e configuração

In [ ]:
import os
import re
import json
import anthropic
import chromadb
import urllib.request
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from typing import List, Tuple

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
print("Cliente Anthropic inicializado.")

## 3. Download e extração do PDF

In [ ]:
PDF_URL = "https://fundar.org.br/wp-content/uploads/2021/06/os-sertoes.pdf"
PDF_PATH = "os-sertoes.pdf"

if not os.path.exists(PDF_PATH):
    print("Baixando o PDF...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)

reader = PdfReader(PDF_PATH)
full_text = "".join(page.extract_text() or "" for page in reader.pages)
clean_text = re.sub(r'\s+', ' ', full_text).strip()
print(f"Texto extraído: {len(clean_text):,} caracteres")

## 4. Chunking e indexação inicial (igual ao Naive RAG)

In [ ]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 150) -> List[str]:
    chunks, start = [], 0
    while start < len(text):
        c = text[start:start + chunk_size].strip()
        if c:
            chunks.append(c)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(clean_text, chunk_size=800, overlap=150)
print(f"Total de chunks: {len(chunks)}")

# Bi-encoder para retrieval inicial (rápido, largo)
bi_encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Cross-encoder para reranking (lento, preciso) — multilíngue via mmarco
cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

print("Modelos carregados!")

## 5. Indexação no ChromaDB

In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="rerank_rag_sertoes",
    metadata={"hnsw:space": "cosine"}
)

BATCH_SIZE = 100
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    embeddings = bi_encoder.encode(batch, show_progress_bar=False).tolist()
    ids = [f"chunk_{j}" for j in range(i, i + len(batch))]
    collection.add(documents=batch, embeddings=embeddings, ids=ids)
    if i % 500 == 0:
        print(f"  Indexados {i + len(batch)}/{len(chunks)} chunks...")

print(f"\nIndexação concluída: {collection.count()} documentos.")

## 6. Pipeline Rerank RAG

> **Fluxo de dois estágios:**  
> 1. **Retrieval largo:** bi-encoder recupera top-`k_initial` (ex: 20) candidatos rapidamente  
> 2. **Reranking:** cross-encoder avalia cada par `(query, chunk)` e reordena por relevância real  
> 3. **Geração:** usa apenas top-`k_final` (ex: 5) após reranking

In [ ]:
def retrieve_candidates(query: str, k_initial: int = 20) -> List[str]:
    """Estágio 1: busca por similaridade com bi-encoder (largo)."""
    query_emb = bi_encoder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_emb,
        n_results=k_initial
    )
    return results["documents"][0]

def rerank(query: str, candidates: List[str], k_final: int = 5) -> List[Tuple[str, float]]:
    """Estágio 2: cross-encoder reranking dos candidatos."""
    pairs = [(query, doc) for doc in candidates]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked[:k_final]

def generate_answer(query: str, context_chunks: List[str]) -> str:
    context = "\n\n---\n\n".join(context_chunks)
    prompt = f"""Você é um assistente especializado em literatura brasileira.
Use APENAS o contexto abaixo para responder à pergunta com detalhes.
Se a informação não estiver no contexto, diga que não encontrou.

CONTEXTO:
{context}

PERGUNTA: {query}

RESPOSTA:"""
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

def rerank_rag(query: str, k_initial: int = 20, k_final: int = 5) -> dict:
    """Pipeline completo: retrieve largo → rerank → generate."""
    candidates = retrieve_candidates(query, k_initial=k_initial)
    ranked = rerank(query, candidates, k_final=k_final)
    top_chunks = [chunk for chunk, _ in ranked]
    top_scores = [float(score) for _, score in ranked]
    answer = generate_answer(query, top_chunks)
    return {
        "query": query,
        "rerank_scores": top_scores,
        "answer": answer
    }

print("Pipeline Rerank RAG pronto!")

## 7. Respondendo às 5 questões

In [ ]:
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em Os Sertões? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

results = []
for i, q in enumerate(questions, 1):
    print(f"\n{'='*70}")
    print(f"QUESTÃO {i}: {q}")
    print('='*70)
    result = rerank_rag(q, k_initial=20, k_final=5)
    results.append(result)
    print(f"[Scores reranking: {[round(s, 3) for s in result['rerank_scores']]}]")
    print(result["answer"])

## 8. Comparação visual dos scores de reranking

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, len(results), figsize=(18, 4))
for idx, (ax, r) in enumerate(zip(axes, results)):
    scores = r["rerank_scores"]
    ax.barh(range(len(scores)), scores, color="steelblue")
    ax.set_yticks(range(len(scores)))
    ax.set_yticklabels([f"Chunk {i+1}" for i in range(len(scores))])
    ax.set_title(f"Q{idx+1}", fontsize=10)
    ax.set_xlabel("Rerank Score")
plt.suptitle("Scores do Cross-Encoder por Questão", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("rerank_scores.png", dpi=100, bbox_inches="tight")
plt.show()
print("Gráfico salvo como rerank_scores.png")

## 9. Salvando resultados

In [ ]:
with open("resultados_rerank_rag.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Resultados salvos em resultados_rerank_rag.json")